# 01 - Investigar, crear guion y generar voz

Este notebook procesa un trabajo por ejecucion interactiva. Guarda estos secretos en Colab: `API_BASE_URL` (HTTPS publico), `WORKER_API_TOKEN`, `OPENAI_API_KEY`, `VOICE_STUDIO_URL`, `HF_TOKEN` (solo lectura del Space privado), `S3_ENDPOINT_URL`, `S3_BUCKET`, `S3_ACCESS_KEY_ID`, `S3_SECRET_ACCESS_KEY` y `STORAGE_PUBLIC_BASE_URL`. OmniVoice acepta atributos de voz admitidos, no descripciones de estilo libres.

El resultado sube el audio a almacenamiento publico S3/R2 y crea automaticamente el trabajo de renderizado.

In [ ]:
%pip install -q "scrapegraphai[openai]" openai boto3 requests gradio_client

In [ ]:
import os
from google.colab import userdata

REQUIRED = (
    "API_BASE_URL", "WORKER_API_TOKEN", "OPENAI_API_KEY", "VOICE_STUDIO_URL",
    "HF_TOKEN", "S3_ENDPOINT_URL", "S3_BUCKET",
    "S3_ACCESS_KEY_ID", "S3_SECRET_ACCESS_KEY", "STORAGE_PUBLIC_BASE_URL",
)
for name in REQUIRED:
    value = userdata.get(name)
    if not value:
        raise RuntimeError(f"Configura el secreto {name} en Colab")
    os.environ[name] = value
os.environ["API_BASE_URL"] = os.environ["API_BASE_URL"].rstrip("/")
os.environ["VOICE_STUDIO_URL"] = os.environ["VOICE_STUDIO_URL"].rstrip("/")
if not os.environ["API_BASE_URL"].startswith("https://"):
    raise RuntimeError("API_BASE_URL debe ser HTTPS y accesible desde Colab")
API_HEADERS = {"Authorization": f"Bearer {os.environ['WORKER_API_TOKEN']}"}

In [ ]:
from pathlib import Path
import json
import boto3
import requests
from gradio_client import Client
from openai import OpenAI
from scrapegraphai.graphs import SmartScraperGraph

def process_one_research_job():
    response = requests.post(
        f"{os.environ['API_BASE_URL']}/v1/jobs/next",
        params={"worker": "research"}, headers=API_HEADERS, timeout=30,
    )
    response.raise_for_status()
    job = response.json()
    if job is None:
        print("No hay investigaciones pendientes.")
        return False

    job_id = job["job_id"]
    lease_headers = {**API_HEADERS, "X-Worker-Lease": job["lease_token"]}
    try:
        request = job["request"]
        source_url = request.get("source_url")
        if not source_url:
            raise ValueError("El trabajo no tiene source_url")
        graph = SmartScraperGraph(
            prompt=(
                f"Investiga {request['topic']}. Devuelve hechos verificables, fechas, "
                "cifras, contexto y las fuentes consultadas. Distingue hechos de opiniones."
            ),
            source=source_url,
            config={
                "llm": {"api_key": os.environ["OPENAI_API_KEY"], "model": "openai/gpt-4o-mini"},
                "verbose": False,
                "headless": True,
            },
        )
        research = graph.run()
        research_json = json.loads(json.dumps(research, ensure_ascii=False, default=str))
        script_response = OpenAI().chat.completions.create(
            model="gpt-4o-mini",
            temperature=0.6,
            messages=[
                {"role": "system", "content": "Escribe guiones de video breves, claros y factuales en español. No inventes citas ni datos. Devuelve solo el texto narrado."},
                {"role": "user", "content": f"Tema: {request['topic']}\nEstilo: {request['style']}\nDuracion objetivo: {request['duration_seconds']} segundos\nInvestigacion: {json.dumps(research_json, ensure_ascii=False)}"},
            ],
        )
        script = (script_response.choices[0].message.content or "").strip()
        if not script:
            raise RuntimeError("El modelo no devolvio un guion")

        voice_client = Client(
            os.environ["VOICE_STUDIO_URL"],
            token=os.environ["HF_TOKEN"],
            verbose=False,
        )
        voice_path = voice_client.predict(
            script,
            "female",
            "young adult",
            "moderate pitch",
            api_name="/predict",
        )
        if not isinstance(voice_path, str) or not Path(voice_path).is_file():
            raise RuntimeError("El Space privado no devolvio el archivo de audio")
        voice_bytes = Path(voice_path).read_bytes()
        if not voice_bytes.startswith(b"RIFF"):
            raise RuntimeError("VoiceStudio devolvio un archivo que no es WAV")

        storage = boto3.client(
            "s3", endpoint_url=os.environ["S3_ENDPOINT_URL"],
            aws_access_key_id=os.environ["S3_ACCESS_KEY_ID"],
            aws_secret_access_key=os.environ["S3_SECRET_ACCESS_KEY"],
        )
        audio_key = f"audio/{job_id}.wav"
        storage.put_object(Bucket=os.environ["S3_BUCKET"], Key=audio_key,
                           Body=voice_bytes, ContentType="audio/wav")
        audio_url = f"{os.environ['STORAGE_PUBLIC_BASE_URL'].rstrip('/')}/{audio_key}"
        complete = requests.post(
            f"{os.environ['API_BASE_URL']}/v1/jobs/{job_id}/complete",
            headers=lease_headers,
            json={"output": {"research": research_json, "script": script, "audio_url": audio_url}},
            timeout=60,
        )
        complete.raise_for_status()
        print(f"Investigacion y voz listas; se encolo el render: {job_id}")
    except Exception as error:
        failed = requests.post(
            f"{os.environ['API_BASE_URL']}/v1/jobs/{job_id}/fail",
            headers=lease_headers, json={"error": str(error)[:1900]}, timeout=60,
        )
        failed.raise_for_status()
        raise RuntimeError(f"Trabajo {job_id} marcado como fallido: {error}") from error
    return True

process_one_research_job()